# Introduction to Risk Metrics

**Level:** Beginner  
**Tags:** Risk, Volatility, VaR, Sharpe Ratio, Drawdown  
**Estimated Time:** 90-120 minutes

## Overview

Master the fundamental risk metrics used in quantitative finance. This notebook covers volatility, Value at Risk (VaR), Conditional VaR (CVaR), Sharpe ratio, maximum drawdown, and other essential risk measures.

## Learning Objectives

By the end of this notebook, you will:

✓ Understand and calculate volatility (standard deviation of returns)  
✓ Compute Value at Risk (VaR) using multiple methods  
✓ Calculate Conditional VaR (Expected Shortfall)  
✓ Measure risk-adjusted returns with the Sharpe ratio  
✓ Analyze drawdowns and recovery periods  
✓ Apply risk metrics to portfolio analysis  
✓ Interpret risk metrics in real-world contexts  

## Prerequisites

- Basic Python programming
- Understanding of returns (see `returns-math.ipynb`)
- Basic statistics (mean, standard deviation)
- Familiarity with pandas and numpy

## References

1. Hull, J.C. (2022). *Options, Futures, and Other Derivatives*. Pearson. (Chapters 21-22)
2. Jorion, P. (2007). *Value at Risk: The New Benchmark for Managing Financial Risk*. McGraw-Hill.
3. McNeil, A.J., Frey, R., & Embrechts, P. (2015). *Quantitative Risk Management*. Princeton University Press.
4. Sharpe, W.F. (1994). "The Sharpe Ratio". *Journal of Portfolio Management*.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
np.random.seed(42)

print('Libraries imported successfully!')

## 1. Introduction to Risk Metrics

### Why Risk Metrics Matter

In finance, **return is only half the story**. Two investments might have the same average return, but vastly different risk profiles. Risk metrics help us:

- **Quantify uncertainty** - Measure how much returns can vary
- **Compare investments** - Evaluate risk-adjusted performance  
- **Set limits** - Define acceptable risk levels
- **Allocate capital** - Optimize portfolio construction
- **Comply with regulations** - Meet Basel III, Solvency II requirements
- **Communicate risk** - Explain exposure to stakeholders

### Key Risk Metrics We'll Cover

1. **Volatility (σ)** - Standard deviation of returns
2. **Value at Risk (VaR)** - Maximum expected loss at a confidence level
3. **Conditional VaR (CVaR)** - Expected loss beyond VaR threshold
4. **Sharpe Ratio** - Risk-adjusted return metric
5. **Maximum Drawdown** - Largest peak-to-trough decline
6. **Sortino Ratio** - Downside risk-adjusted return
7. **Beta** - Systematic risk relative to market

### Real-World Applications

**Portfolio Management:**
- Asset allocation decisions
- Performance attribution
- Manager evaluation

**Risk Management:**
- Setting position limits
- Calculating regulatory capital
- Stress testing

**Trading:**
- Position sizing
- Stop-loss placement
- Strategy evaluation

### A Historical Perspective

Risk management evolved significantly after major financial events:
- **1987 Black Monday** → Focus on tail risk
- **1998 LTCM Crisis** → Importance of liquidity risk  
- **2008 Financial Crisis** → Systemic risk awareness
- **2020 COVID Crash** → Rapid regime changes

Let's dive into each metric!

In [ ]:
# Generate sample data for demonstration
np.random.seed(42)

# Simulate daily returns for multiple assets over 2 years (504 trading days)
n_days = 504
dates = pd.date_range(start='2023-01-01', periods=n_days, freq='B')

# Asset 1: Low-risk (e.g., bond fund)
returns_low_risk = np.random.normal(0.0003, 0.005, n_days)  # 7.5% annual return, 8% vol

# Asset 2: Medium-risk (e.g., diversified stock portfolio)
returns_med_risk = np.random.normal(0.0005, 0.012, n_days)  # 12.5% annual return, 19% vol

# Asset 3: High-risk (e.g., tech stock)
returns_high_risk = np.random.normal(0.0006, 0.025, n_days)  # 15% annual return, 40% vol

# Create DataFrame
df = pd.DataFrame({
    'Low_Risk': returns_low_risk,
    'Medium_Risk': returns_med_risk,
    'High_Risk': returns_high_risk
}, index=dates)

# Calculate cumulative returns and prices
for col in df.columns:
    df[f'{col}_Price'] = 100 * (1 + df[col]).cumprod()
    df[f'{col}_Cumulative'] = (1 + df[col]).cumprod() - 1

print("Sample Return Data Generated")
print("=" * 70)
print(f"\nPeriod: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Trading Days: {n_days} ({n_days/252:.1f} years)\n")

# Display first few rows
print("First 5 days of returns:")
print(df[['Low_Risk', 'Medium_Risk', 'High_Risk']].head())

print("\nReturn Statistics:")
print(df[['Low_Risk', 'Medium_Risk', 'High_Risk']].describe())

# Quick visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Price evolution
for col in ['Low_Risk_Price', 'Medium_Risk_Price', 'High_Risk_Price']:
    axes[0].plot(df.index, df[col], label=col.replace('_Price', ''), linewidth=2, alpha=0.8)
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Initial Price')
axes[0].set_title('Price Evolution: Three Risk Profiles', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (Initial = $100)', fontsize=11)
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Returns distribution
colors = ['green', 'blue', 'red']
for i, col in enumerate(['Low_Risk', 'Medium_Risk', 'High_Risk']):
    axes[1].hist(df[col], bins=50, alpha=0.5, label=col.replace('_', ' '), 
                 edgecolor='black', color=colors[i])
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Return Distributions', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Daily Return', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nFinal Prices:")
for col in ['Low_Risk', 'Medium_Risk', 'High_Risk']:
    final_price = df[f'{col}_Price'].iloc[-1]
    total_return = df[f'{col}_Cumulative'].iloc[-1]
    print(f"  {col:15}: ${final_price:7.2f} (Total Return: {total_return:7.2%})")

## 2. Mathematical Foundation

### 2.1 Volatility (Standard Deviation)

**Definition:** Volatility measures the dispersion of returns around the mean. It's the square root of variance.

$$\sigma = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N} (r_i - \bar{r})^2}$$

Where:
- $\sigma$ = volatility (standard deviation)
- $r_i$ = return in period $i$
- $\bar{r}$ = mean return
- $N$ = number of observations

**Annualization:** For daily returns, annualized volatility is:

$$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$$

The $\sqrt{252}$ factor assumes 252 trading days per year and **independent, identically distributed (IID) returns**.

**Properties:**
- ✓ **Symmetric** - Treats upside and downside equally
- ✓ **Scale-independent** - Can compare different assets
- ✗ **Assumes normality** - May underestimate tail risk
- ✗ **Backward-looking** - Historical, not predictive

---

### 2.2 Value at Risk (VaR)

**Definition:** VaR answers: "What is the maximum loss over a given time horizon at a given confidence level?"

$$\text{VaR}_{\alpha} = \inf\{x : P(L > x) \leq 1 - \alpha\}$$

Or equivalently:
$$P(L \leq \text{VaR}_{\alpha}) = \alpha$$

Where:
- $L$ = loss (negative return)
- $\alpha$ = confidence level (e.g., 95%, 99%)
- $\text{VaR}_{\alpha}$ = Value at Risk at confidence level $\alpha$

**Example:** 
- 95% 1-day VaR = \$1 million means:
- "We are 95% confident we won't lose more than \$1M tomorrow"
- Or: "There's a 5% chance of losing more than \$1M tomorrow"

**Three Methods to Calculate VaR:**

**1. Historical Method** (Non-parametric):
$$\text{VaR}_{\alpha} = -\text{Percentile}(returns, 1-\alpha)$$

**2. Parametric Method** (Variance-Covariance):
$$\text{VaR}_{\alpha} = -(\mu + z_{\alpha} \sigma)$$

Where $z_{\alpha}$ is the z-score for confidence level $\alpha$:
- 95% → $z_{0.95} = -1.645$
- 99% → $z_{0.99} = -2.326$

**3. Monte Carlo Simulation**:
- Simulate many possible return scenarios
- Calculate percentile from simulated distribution

---

### 2.3 Conditional VaR (CVaR / Expected Shortfall)

**Definition:** CVaR is the expected loss given that the loss exceeds VaR. It answers: "If things get bad (beyond VaR), how bad on average?"

$$\text{CVaR}_{\alpha} = E[L | L > \text{VaR}_{\alpha}]$$

Or for discrete data:
$$\text{CVaR}_{\alpha} = -\frac{1}{(1-\alpha)N} \sum_{i: r_i \leq \text{VaR}_{\alpha}} r_i$$

**Properties:**
- ✓ **Coherent risk measure** (satisfies subadditivity)
- ✓ **Tail risk sensitive** - Captures severity beyond VaR
- ✓ **Preferred by regulators** (Basel Committee)
- Always $\text{CVaR}_{\alpha} \geq \text{VaR}_{\alpha}$

---

### 2.4 Sharpe Ratio

**Definition:** The Sharpe ratio measures excess return per unit of risk.

$$\text{Sharpe Ratio} = \frac{\bar{r} - r_f}{\sigma}$$

For annualized returns:
$$\text{Sharpe Ratio}_{annual} = \frac{r_{annual} - r_f}{\sigma_{annual}} = \frac{\bar{r}_{daily} \times 252 - r_f}{\sigma_{daily} \times \sqrt{252}}$$

Where:
- $\bar{r}$ = average return
- $r_f$ = risk-free rate (e.g., T-bill rate)
- $\sigma$ = volatility

**Interpretation:**
- SR > 1.0: Excellent (very good risk-adjusted returns)
- SR > 0.5: Good
- SR > 0: Positive (beats risk-free rate)
- SR < 0: Negative (underperforms risk-free rate)

**Limitations:**
- Assumes normal distribution
- Penalizes upside volatility equally with downside
- Can be manipulated by changing sampling frequency

---

### 2.5 Maximum Drawdown

**Definition:** The maximum peak-to-trough decline over a specified period.

$$\text{MDD} = \max_{t} \left( \frac{\text{Peak}_t - \text{Trough}_{t}}{\text{Peak}_t} \right)$$

More formally:
$$\text{MDD} = \max_{0 \leq s \leq t \leq T} \left( \frac{V_s - V_t}{V_s} \right)$$

Where:
- $V_t$ = portfolio value at time $t$
- Peak = maximum value up to time $t$

**Related Metrics:**
- **Drawdown Duration**: Time from peak to recovery
- **Calmar Ratio**: Annual return / Maximum drawdown

---

### 2.6 Sortino Ratio

**Definition:** Like Sharpe ratio, but only penalizes downside volatility.

$$\text{Sortino Ratio} = \frac{\bar{r} - r_f}{\sigma_{downside}}$$

Where downside deviation:
$$\sigma_{downside} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \min(r_i - r_f, 0)^2}$$

**Advantage over Sharpe:**
- Doesn't penalize upside volatility
- Better for non-symmetric return distributions
- Preferred for strategies with positive skew

---

### 2.7 Beta (Systematic Risk)

**Definition:** Measures an asset's sensitivity to market movements.

$$\beta = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)} = \frac{\sigma_{i,m}}{\sigma_m^2}$$

Where:
- $r_i$ = asset return
- $r_m$ = market return
- $\sigma_{i,m}$ = covariance between asset and market
- $\sigma_m^2$ = variance of market

**Interpretation:**
- $\beta = 1$: Moves with market
- $\beta > 1$: More volatile than market (aggressive)
- $\beta < 1$: Less volatile than market (defensive)  
- $\beta = 0$: Uncorrelated with market

**From CAPM:**
$$E[r_i] = r_f + \beta_i (E[r_m] - r_f)$$

## 3. Python Implementation

### 3.1 Volatility Calculations

In [ ]:
# Calculate volatility for all assets
print("Volatility Analysis")
print("=" * 70)

risk_metrics = {}

for col in ['Low_Risk', 'Medium_Risk', 'High_Risk']:
    # Daily volatility
    daily_vol = df[col].std()
    
    # Annualized volatility (assuming 252 trading days)
    annual_vol = daily_vol * np.sqrt(252)
    
    # Store results
    risk_metrics[col] = {
        'daily_vol': daily_vol,
        'annual_vol': annual_vol
    }
    
    print(f"\n{col}:")
    print(f"  Daily Volatility:      {daily_vol:.6f} ({daily_vol*100:.4f}%)")
    print(f"  Annualized Volatility: {annual_vol:.6f} ({annual_vol*100:.2f}%)")

# Rolling volatility
window = 60  # 60-day rolling window
fig, ax = plt.subplots(figsize=(14, 6))

for col in ['Low_Risk', 'Medium_Risk', 'High_Risk']:
    rolling_vol = df[col].rolling(window=window).std() * np.sqrt(252)
    ax.plot(df.index, rolling_vol * 100, label=col.replace('_', ' '), linewidth=2, alpha=0.8)

ax.set_title(f'{window}-Day Rolling Volatility (Annualized)', fontsize=14, fontweight='bold')
ax.set_ylabel('Volatility (%)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Visualization and Analysis

## 5. Practical Applications

## Summary

This notebook covered intro to risk metrics. Key takeaways:

- Practical implementation with Python
- Visualizations and interpretations
- Real-world applications
- Best practices and pitfalls

### Next Steps

- Practice with real market data
- Explore related topics
- Build your own variations

### Additional Resources

- [QuantEdX Courses](https://www.quantedx.com/courses)
- [Community Forum](https://www.quantedx.com/community)
- [GitHub Repository](https://github.com/quantedx)